In [47]:
import pandas as pd

In [ ]:
# Load integrated training data (2014-2021) and prescription data
df = pd.read_csv('../dataset/integrated_data.csv')
rx = pd.read_csv('../dataset/prescription.csv')

# Drop auto-generated index columns
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
if 'Unnamed: 0' in rx.columns:
    rx = rx.drop(columns=['Unnamed: 0'])

print("Integrated data shape:", df.shape)
print("Prescription data shape:", rx.shape)

In [ ]:
# Keep only suggested prescription features
rx = rx[['Observation_ID', 'Quantity', 'Form', 'Strength', 'Day_Supply']]

print("Prescription selected shape:", rx.shape)
print("Missing values:\n", rx.isnull().sum())

Prescription selected shape: (905728, 5)
Missing values:
 Observation_ID        0
Quantity          97497
Form              97497
Strength          97524
Day_Supply        97497
dtype: int64


In [ ]:
# Join prescription features onto integrated data by Observation_ID
super_df = df.merge(rx, on='Observation_ID', how='left')

print("Super dataset shape:", super_df.shape)
print("Columns:", super_df.columns.tolist())
print("\nMissing values:\n", super_df.isnull().sum())

Super dataset shape: (905728, 11)
Columns: ['Observation_ID', 'Drug', 'Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity', 'Quantity', 'Form', 'Strength', 'Day_Supply']

Missing values:
 Observation_ID            0
Drug                      0
Age                       0
Sex                       0
Family_income             0
Insurance_coverage        0
Race_ethnicity            0
Quantity              97497
Form                  97497
Strength              97524
Day_Supply            97497
dtype: int64


In [ ]:
# Verify all missing values come from 'no prescriptions' rows only
no_rx_mask = super_df['Drug'] == 'no prescriptions'

print(f"Total 'no prescriptions' rows: {no_rx_mask.sum()}")
print()
for col in ['Quantity', 'Form', 'Strength', 'Day_Supply']:
    missing_in_no_rx = super_df.loc[no_rx_mask, col].isnull().sum()
    missing_in_actual = super_df.loc[~no_rx_mask, col].isnull().sum()
    print(f"{col}:")
    print(f"  Missing in 'no prescriptions' rows: {missing_in_no_rx}")
    print(f"  Missing in actual drug rows:        {missing_in_actual}")

Total 'no prescriptions' rows: 97497

Quantity:
  Missing in 'no prescriptions' rows: 97497
  Missing in actual drug rows:        0
Form:
  Missing in 'no prescriptions' rows: 97497
  Missing in actual drug rows:        0
Strength:
  Missing in 'no prescriptions' rows: 97497
  Missing in actual drug rows:        27
Day_Supply:
  Missing in 'no prescriptions' rows: 97497
  Missing in actual drug rows:        0


In [ ]:
# Check which drugs have missing Strength in actual drug rows
missing_strength = super_df[
    (super_df['Drug'] != 'no prescriptions') & 
    (super_df['Strength'].isnull())
]

print(f"Total rows with missing Strength in actual drugs: {len(missing_strength)}")
print(f"\nDrugs with missing Strength:")
print(missing_strength['Drug'].value_counts())

Total rows with missing Strength in actual drugs: 27

Drugs with missing Strength:
Drug
hydrocodone    21
albuterol       6
Name: count, dtype: int64


In [ ]:
# Fill 27 missing Strength values with median Strength per drug
# Only 27 rows (0.003%) — median per drug is the most defensible approach
for drug in ['hydrocodone', 'albuterol']:
    median_strength = super_df.loc[
        (super_df['Drug'] == drug) & (super_df['Strength'].notnull()), 
        'Strength'
    ].median()
    
    mask = (super_df['Drug'] == drug) & (super_df['Strength'].isnull())
    super_df.loc[mask, 'Strength'] = median_strength
    print(f"{drug}: filled {mask.sum()} rows with median Strength = {median_strength}")

print(f"\nMissing Strength in actual drug rows after fix: {super_df.loc[super_df['Drug'] != 'no prescriptions', 'Strength'].isnull().sum()}")

hydrocodone: filled 21 rows with median Strength = 325000.0
albuterol: filled 6 rows with median Strength = 108.0

Missing Strength in actual drug rows after fix: 0


In [ ]:
# Final summary of super dataset
print("FINAL SUPER DATASET SUMMARY")
print(f"Shape: {super_df.shape}")
print(f"\nColumns: {super_df.columns.tolist()}")
print(f"\nMissing values:")
print(super_df.isnull().sum())
print(f"\nUnique drugs: {super_df['Drug'].nunique()}")
print(f"No prescriptions rows: {(super_df['Drug'] == 'no prescriptions').sum()}")
print(f"Actual drug rows: {(super_df['Drug'] != 'no prescriptions').sum()}")
print(f"\nSample:")
print(super_df.head(3))

=== FINAL SUPER DATASET SUMMARY ===
Shape: (905728, 11)

Columns: ['Observation_ID', 'Drug', 'Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity', 'Quantity', 'Form', 'Strength', 'Day_Supply']

Missing values:
Observation_ID            0
Drug                      0
Age                       0
Sex                       0
Family_income             0
Insurance_coverage        0
Race_ethnicity            0
Quantity              97497
Form                  97497
Strength              97497
Day_Supply            97497
dtype: int64

Unique drugs: 217
No prescriptions rows: 97497
Actual drug rows: 808231

Sample:
   Observation_ID        Drug   Age     Sex  Family_income Insurance_coverage  \
0               1  clobetasol  36.0    Male        68000.0            Private   
1               2  clobetasol  33.0  Female         3500.0             Public   
2               3  clobetasol  27.0  Female        55000.0            Private   

       Race_ethnicity  Quantity     Form  Str

In [ ]:
# Save super dataset
super_df.to_csv('super_integrated_data.csv', index=False)
print("Saved: super_integrated_data.csv")
print(f"Shape: {super_df.shape}")
print(f"\nFeatures:")
print("  Demographics: Age, Sex, Family_income, Insurance_coverage, Race_ethnicity")
print("  Prescription: Quantity, Form, Strength, Day_Supply")
print("  Target:       Drug")
print("  Identifier:   Observation_ID")

Saved: super_integrated_data.csv
Shape: (905728, 11)

Features:
  Demographics: Age, Sex, Family_income, Insurance_coverage, Race_ethnicity
  Prescription: Quantity, Form, Strength, Day_Supply
  Target:       Drug
  Identifier:   Observation_ID

Missing values (all from 'no prescriptions' rows):
  Quantity, Form, Strength, Day_Supply: 97,497 rows (10.8%)

Note: Missing values are intentional — absence of prescription
      data is itself a feature for 'no prescriptions' class
      XGBoost handles NaN natively
      RealMLP: fill with -1 as sentinel value
